# Module 4: Pretraining

This notebook covers the core pretraining loop - how we train a language model from scratch.

## What You'll Learn

- **Training loop** - Forward, backward, update cycle
- **Muon optimizer** - Better than AdamW for matrix weights
- **Learning rate scheduling** - Warmup and cooldown
- **Distributed training (DDP)** - Multi-GPU parallelism
- **Model Flop Utilization (MFU)** - Measuring efficiency

In [ ]:
import os
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 4.1 The Training Loop

The core training loop is simple:

```python
for step in range(num_iterations):
    # 1. Get batch of data
    x, y = next(data_loader)
    
    # 2. Forward pass: compute loss
    loss = model(x, y)
    
    # 3. Backward pass: compute gradients
    loss.backward()
    
    # 4. Update weights
    optimizer.step()
    optimizer.zero_grad()
```

In [ ]:
# Simple training example
class TinyModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.linear = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, targets=None):
        h = self.embed(x)
        logits = self.linear(h)
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return loss
        return logits

# Create model
vocab_size = 1000
model = TinyModel(vocab_size, 256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Training loop
losses = []
for step in range(100):
    # Random data for demo
    x = torch.randint(0, vocab_size, (32, 64), device=device)
    y = torch.randint(0, vocab_size, (32, 64), device=device)
    
    # Forward
    loss = model(x, y)
    losses.append(loss.item())
    
    # Backward
    loss.backward()
    
    # Update
    optimizer.step()
    optimizer.zero_grad()

import matplotlib.pyplot as plt
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Random baseline (log vocab_size): {math.log(vocab_size):.4f}")

## 4.2 Gradient Accumulation

When the desired batch size doesn't fit in GPU memory, we accumulate gradients over multiple micro-batches:

In [ ]:
# Gradient accumulation example
grad_accum_steps = 4
device_batch_size = 8  # Smaller batch that fits in memory
effective_batch_size = device_batch_size * grad_accum_steps  # = 32

print(f"Device batch size: {device_batch_size}")
print(f"Gradient accumulation steps: {grad_accum_steps}")
print(f"Effective batch size: {effective_batch_size}")

# Reset model
model = TinyModel(vocab_size, 256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10):
    total_loss = 0
    
    # Accumulate gradients
    for micro_step in range(grad_accum_steps):
        x = torch.randint(0, vocab_size, (device_batch_size, 64), device=device)
        y = torch.randint(0, vocab_size, (device_batch_size, 64), device=device)
        
        loss = model(x, y)
        loss = loss / grad_accum_steps  # Normalize loss
        loss.backward()  # Gradients accumulate!
        total_loss += loss.item()
    
    # Update after accumulation
    optimizer.step()
    optimizer.zero_grad()
    
    print(f"Step {step}: loss = {total_loss:.4f}")

## 4.3 The Muon Optimizer

Nanochat uses **Muon** for matrix weights and **AdamW** for embeddings.

Muon is specialized for matrices and uses orthogonalization to maintain better gradient directions.

In [ ]:
# Simplified Muon implementation
class SimpleMuon(torch.optim.Optimizer):
    """
    Muon optimizer - uses Newton-Schulz orthogonalization.
    Works best for matrix weights (Linear layers).
    """
    def __init__(self, params, lr=0.02, momentum=0.95):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)
    
    def step(self):
        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            
            for p in group['params']:
                if p.grad is None:
                    continue
                
                g = p.grad
                state = self.state[p]
                
                # Initialize momentum buffer
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(g)
                
                buf = state['momentum_buffer']
                buf.mul_(momentum).add_(g)
                
                # Newton-Schulz orthogonalization (simplified)
                # This makes the update more "orthogonal"
                if buf.dim() == 2:
                    # For 2D weights, normalize by spectral norm approximation
                    update = buf / (buf.norm() + 1e-8) * g.norm()
                else:
                    update = buf
                
                p.data.add_(update, alpha=-lr)

print("Muon vs AdamW in nanochat:")
print("  - Muon: Used for transformer block matrices (attention, MLP)")
print("  - AdamW: Used for embeddings and lm_head")
print("")
print("Why two optimizers?")
print("  - Muon works better for high-dimensional matrices")
print("  - AdamW works better for embeddings")
print("  - Combined, they achieve better performance")

## 4.4 Learning Rate Scheduling

Nanochat uses a schedule with:
- **Warmup**: Gradually increase LR from 0
- **Constant**: Full LR during main training
- **Cooldown**: Gradually decrease LR to 0

In [ ]:
def get_lr_multiplier(step, num_iterations, warmup_ratio=0.0, warmdown_ratio=0.2, final_lr_frac=0.0):
    """Nanochat's learning rate schedule."""
    warmup_iters = round(warmup_ratio * num_iterations)
    warmdown_iters = round(warmdown_ratio * num_iterations)
    
    if step < warmup_iters:
        # Linear warmup
        return (step + 1) / warmup_iters
    elif step <= num_iterations - warmdown_iters:
        # Constant
        return 1.0
    else:
        # Linear cooldown
        progress = (num_iterations - step) / warmdown_iters
        return progress * 1.0 + (1 - progress) * final_lr_frac

# Visualize the schedule
num_iterations = 1000
steps = range(num_iterations)
lr_mults = [get_lr_multiplier(s, num_iterations, warmup_ratio=0.05, warmdown_ratio=0.2) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lr_mults)
plt.xlabel('Step')
plt.ylabel('LR Multiplier')
plt.title('Learning Rate Schedule (5% warmup, 20% cooldown)')
plt.axvline(x=50, color='g', linestyle='--', alpha=0.5, label='End warmup')
plt.axvline(x=800, color='r', linestyle='--', alpha=0.5, label='Start cooldown')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Momentum warmup for Muon
def get_muon_momentum(step):
    """Muon momentum warmup from 0.85 to 0.95 over 300 steps."""
    frac = min(step / 300, 1)
    return (1 - frac) * 0.85 + frac * 0.95

steps = range(500)
momentums = [get_muon_momentum(s) for s in steps]

plt.figure(figsize=(8, 4))
plt.plot(steps, momentums)
plt.xlabel('Step')
plt.ylabel('Momentum')
plt.title('Muon Momentum Warmup')
plt.axhline(y=0.85, color='r', linestyle='--', alpha=0.5, label='Initial (0.85)')
plt.axhline(y=0.95, color='g', linestyle='--', alpha=0.5, label='Final (0.95)')
plt.axvline(x=300, color='gray', linestyle='--', alpha=0.5, label='Warmup end')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4.5 Distributed Data Parallel (DDP)

For multi-GPU training, PyTorch's DDP synchronizes gradients across GPUs.

In [ ]:
# DDP setup (conceptual - requires multiple GPUs)
def setup_ddp():
    """
    Setup for Distributed Data Parallel.
    In practice, this is launched with torchrun.
    """
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    
    # These are set by torchrun
    rank = int(os.environ.get('RANK', 0))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    
    # Initialize process group
    dist.init_process_group(backend='nccl')
    
    # Set device for this rank
    torch.cuda.set_device(local_rank)
    device = torch.device(f'cuda:{local_rank}')
    
    return rank, local_rank, world_size, device

print("DDP Workflow:")
print("  1. Launch with: torchrun --nproc_per_node=8 script.py")
print("  2. Each GPU gets a separate process")
print("  3. Each process loads different data shards")
print("  4. Forward pass runs independently")
print("  5. Gradients are synchronized (all-reduce)")
print("  6. All GPUs update with same gradients")

print("\nNanochat command:")
print("  torchrun --nproc_per_node=8 -m scripts.base_train -- --depth=20")

## 4.6 Model FLOPs Utilization (MFU)

MFU measures how efficiently we use the GPU's theoretical compute capacity.

In [ ]:
def estimate_flops_per_token(n_params, n_layer, n_head, head_dim, seq_len):
    """
    Estimate FLOPs per token.
    Reference: https://arxiv.org/abs/2204.02311
    """
    # Approximate formula:
    # 6 * N_params (for forward + backward on weights)
    # + 12 * L * H * D * T (for attention computation)
    flops = 6 * n_params + 12 * n_layer * n_head * head_dim * seq_len
    return flops

def compute_mfu(flops_per_token, batch_size, time_per_step, gpu_tflops=989, num_gpus=8):
    """
    Compute Model FLOPs Utilization.
    H100 SXM: 989 TFLOPS (bfloat16, without sparsity)
    """
    total_flops = flops_per_token * batch_size
    achieved_tflops = total_flops / time_per_step / 1e12
    theoretical_tflops = gpu_tflops * num_gpus
    mfu = achieved_tflops / theoretical_tflops * 100
    return mfu, achieved_tflops, theoretical_tflops

# Example for nanochat depth=20
n_params = 561_000_000  # 561M params
n_layer = 20
n_head = 10
head_dim = 128
seq_len = 2048
batch_size = 524288  # tokens per step
time_per_step = 2.5  # seconds (example)

flops_per_token = estimate_flops_per_token(n_params, n_layer, n_head, head_dim, seq_len)
mfu, achieved, theoretical = compute_mfu(flops_per_token, batch_size, time_per_step)

print(f"Nanochat d=20 Performance Estimate:")
print(f"  - Parameters: {n_params/1e6:.0f}M")
print(f"  - FLOPs per token: {flops_per_token:.2e}")
print(f"  - Batch size: {batch_size:,} tokens")
print(f"  - Time per step: {time_per_step}s")
print(f"  - Achieved: {achieved:.1f} TFLOPS")
print(f"  - Theoretical (8xH100): {theoretical:.0f} TFLOPS")
print(f"  - MFU: {mfu:.1f}%")

print("\nTypical MFU ranges:")
print("  - 30-40%: Good")
print("  - 40-50%: Very good")
print("  - 50%+: Excellent (close to theoretical max)")

## 4.7 Training Dynamics

Key metrics to monitor during training:

In [ ]:
# Simulated training metrics
import numpy as np

steps = np.arange(10000)
train_loss = 10 * np.exp(-steps/2000) + 2.5 + np.random.randn(len(steps)) * 0.1
val_loss = 10 * np.exp(-steps/2000) + 2.7 + np.random.randn(len(steps)) * 0.05
val_bpb = val_loss / np.log(2)  # bits per byte ≈ loss / ln(2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Training loss
axes[0].plot(steps, train_loss, alpha=0.7, label='Train')
axes[0].plot(steps, val_loss, alpha=0.7, label='Val')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bits per byte
axes[1].plot(steps, val_bpb)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Bits per Byte')
axes[1].set_title('Validation Bits per Byte')
axes[1].grid(True, alpha=0.3)

# Learning rate
lr = [get_lr_multiplier(s, 10000, 0.0, 0.2) for s in steps]
axes[2].plot(steps, lr)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('LR Multiplier')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key metrics:")
print(f"  - Train loss: Smoothed version of batch losses")
print(f"  - Val loss: Evaluated on held-out data")
print(f"  - Bits per byte: How many bits needed to encode each byte")
print(f"    (lower is better, 1.0 = perfect compression)")

## Summary

In this notebook, we learned:

1. ✅ **Training loop** - Forward, backward, update cycle
2. ✅ **Gradient accumulation** - For large effective batch sizes
3. ✅ **Muon optimizer** - Specialized for matrix weights
4. ✅ **LR scheduling** - Warmup and cooldown
5. ✅ **Distributed training** - Multi-GPU with DDP
6. ✅ **MFU** - Measuring GPU utilization

## Next Steps

Continue to **[Module 5: Mid-training and SFT](05_finetuning.ipynb)** to learn:
- Mid-training on conversation data
- Supervised fine-tuning for chat
- Training masks for conversation data

---

**Estimated time for this notebook: 45-60 minutes**